In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL='gpt-6-luna'

In [2]:
from openai import OpenAI


client = OpenAI()

In [3]:
response = client.responses.create(
    model=MODEL,
    input="2026년 11월 11일 삼성전자 주식 가격 알려줘",   
)
print(response.output_text)

2026년 11월 11일 삼성전자 주가는 아직 확정되지 않았습니다. 주가를 예측해 드릴 수는 있지만, 실제 가격은 해당 날짜의 시장 거래가 끝난 뒤 확인할 수 있습니다. 전망을 원하시면 알려 주세요.


#### Function Calling 

- AI 모델이 사용자가 요청한 작업을 직접 처리할 수 없을 때 외부에서 미리 정의한 함수를 호출하도록 요청

- 1. 함수정의
- 2. 함수호출 - AI가 함수호출을 요청
- 3. 함수실행 - 개발자의 프로그램이 함수를 실행

In [5]:
response = client.responses.create(
    model=MODEL,
    input="10과 20을 더해줘",   
)
print(response.output_text)

10과 20의 합은 30입니다.


In [4]:
# 함수 정의
def add(a, b):
    return a + b

In [8]:
# 함수 의미 작성
tools = [
    {
        "type":"function",
        "name":"add",
        "description":"두 숫자를 더하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                }
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    }
]

In [9]:
response = client.responses.create(
    model=MODEL,
    input="10과 20을 더해줘", 
    tools=tools  
)
print(response.output)

[ResponseFunctionToolCall(arguments='{"a":10,"b":20}', call_id='call_0iEUyQiC3WraqsZLKmuHnwqY', name='add', type='function_call', id='fc_08d09972a9709389006af57bd74ce887d2877a28d11afea844', async_=None, caller=None, namespace=None, status='completed')]


In [10]:
# 실제 함수 실행 
import json

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        # 함수 호출
        result = add(args['a'],args['b'])
        print(result)


30


In [11]:
response = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=[
        {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":str(result)
        }
    ] 
)
print(response.output_text)

10 + 20 = 30


### 사용자의 입력에 따라 다른 함수 호출
- 100 에서 30을 빼고, 20을 곱해줘

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b

In [6]:
tools = [
    {
        "type":"function",
        "name":"add",
        "description":"두 숫자를 더하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                }
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
    {
        "type":"function",
        "name":"subtract",
        "description":"첫 번째 숫자에서 두 번째 숫자를 뺀다.",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                }
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
    {
        "type":"function",
        "name":"multiply",
        "description":"두 숫자를 곱하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                }
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
    {
        "type":"function",
        "name":"divide",
        "description":"첫 번째 숫자를 두 번째 숫자로 나눈다.",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                }
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    }
]

In [14]:
import json

response = client.responses.create(
    model=MODEL,
    input="100에서 30을 빼고, 그 결과에 20을 곱해줘", 
    tools=tools  
)
print(response.output)

# function call 이 있으면 함수 실행
function_registry = {
    "add":add,
    "subtract":subtract,
    "multiply":multiply,
    "divide":divide,
}

# for item in response.output:
#     if item.type == "function_call":
#         args = json.loads(item.arguments)
#         # 함수 호출
#         result = add(args['a'],args['b'])
#         print(result)

while True:

    # function_call 이 있는지 확인
    function_calls = [item  for item in response.output if item.type == "function_call"]

    # function_call 이 없다면 반복문 종료
    if not function_calls:
        break

    # function_calls 있는 경우
    tool_outputs = []
    for item in function_calls:
        # AI가 전달한 인자 가져오기
        args = json.loads(item.arguments)
        # AI가 요청한 함수 이름
        func = function_registry[item.name]
        # func 실행
        result = func(**args)

        tool_outputs.append(
            {
                "type":"function_call_output",
                "call_id":item.call_id,
                "output":str(result)
            }
        )

    
    response = client.responses.create(
        model=MODEL,
        previous_response_id=response.id,
        input=tool_outputs,  
        tools=tools  
    )

print(response.output_text)

[ResponseFunctionToolCall(arguments='{"a":100,"b":30}', call_id='call_3Km4puxLxQYnQyYwb4JHwaND', name='subtract', type='function_call', id='fc_0c31c598d6adb27f006af6688a453087d299b3ee31bae5f663', async_=None, caller=None, namespace=None, status='completed')]
결과는 **1,400**입니다.


In [ ]:
documents = [
    {
        "title":"AI 기술 보고서",
        "content":"LLM 과 RAG 시스템의 구조를 설명한다."
    },
    {
        "title":"보안 기술 보고서",
        "content":"PQC와 양자내성암호를 설명한다."
    },
    {
        "title":"추천 시스템 보고서",
        "content":"사용자 행동 데이터를 이용한 추천 모델이다."
    },
]

def search_documents(keyword):
    results = []

    for doc in documents:
        if keyword.lower() in (doc['title'] + doc['content']).lower():
            results.append(doc)

    return results

tools = [
    {
        "type":"function",
        "name":"search_documents",
        "description":"회사 내부 문서를 키워드로 검색한다.",
        "parameters":{
            "type":"object",
            "properties":{
                "keyword":{
                    "type":"string",
                    "description":"검색 키워드"
                },               
            },
            "required":["keyword"],
            "additionalProperties":False
        },
        "strict":True
    }
]

response = client.responses.create(
    model=MODEL,
    input="AI", 
    tools=tools  
)

print(response.output)

function_calls = [item  for item in response.output if item.type == "function_call"]

tool_outputs = []
for item in function_calls:
    # AI가 전달한 인자 가져오기
    args = json.loads(item.arguments) 
    # func 실행
    result = search_documents(**args)

    tool_outputs.append(
        {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":json.dumps(result,ensure_ascii=False) # dict 구조일때
        }
    )


response = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=tool_outputs,  
    tools=tools  
)

print(response.output_text)

[ResponseReasoningItem(id='rs_0008fd05f1b1b974006af66c7550a087d2927baaa0053489e6', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq9mx2G7I_VwCRpLco8rs4FDguj8puumQAQ6Sd1IvLwk1NTBYBjkqvQ4mJunYhGIF7XFi-3P3jz_dIh0iMVfjFEAyb__mdA3LgBGgksN51WSPLjqEDcYABbqzwdSugBf04fMUlZ4imBQx6OqHrwj43_PVtD93xkQgn46z9BIEzowQqCmF43cWD3RMTj-BkDkAtW_aAPNC63aQl-ix6WuKd7ypwN0r7UFf0vstz31z-bldIOIWudITNCzaTzTR8Slr8kcuIbfQRBRSoBZoQUa0wxDtJ6oWi_ZVcZ-2SM8EMSTBiuL5nYYCVEECek7OQSmhucu-W-6LOMojqE-pPEKelhTFgVfxiCXWBx2Af0xjhIAlauPiltPpPPoTvHjVGUafzqvLge55BHjgF5agh01JC0gSWoUwTy-Lk1N3TH3Zn3t87CQ3RC51eVbyjrlFFOXMtWmQU_JZNdC6dE30n24oPg0aRuN1vxBKA3BeUjYsYL0_1TNQ513vdyj_OsFWbqhevBfn5kbCWVuFpEFFsig5rX3avB8BhjoQd0s4ae8t5RfuWFjd9auR-y9APSZxbG4Et6-V-jplv3Ltf2sjrubgOem3HdZUuQVcluOjj6z5aOMzfGmn39AlVD-OV6McAeDIQ3PbZutneZq--_6YRnSx9QD2FrXhAPHDnMMB93sB624rWPwSGleaNr36QDqGxfOLLeGBIvPt64wD9RcAWjwOCZxgtU_isTlxgr3V1mxV8GnI8hY5qWONGfgDtuENXUpdf4sp4sJ8Co0FyMhQodZyMB4-VtLj8EL5kAX_6JkyBovnKft89t3s6U5GMm_TWD_JDUS31dMPslAQpCQgp

In [17]:
# uv add yfinance

import yfinance as yf

stock = yf.Ticker("005930.KS")
#data = stock.history(start="2026-11-09")
data = stock.history(period="1d") # 1mo
print(data)

                               Open      High       Low     Close   Volume  \
Date                                                                         
2026-11-13 00:00:00+09:00  214000.0  216000.0  212000.0  213250.0  3702905   

                           Dividends  Stock Splits  
Date                                                
2026-11-13 00:00:00+09:00        0.0           0.0  


In [18]:
def get_korean_stock_price(code:str):

    symbol = f"{code}.KS"
    stock = yf.Ticker(symbol)
    data = stock.history(period="1d")

    if data.empty:
        return json.dumps({"error":"주식 데이터를 찾을 수 없습니다."})

    # 마지막 행 가져오기
    latest = data.iloc[-1]

    return json.dumps({
        "code":code,
        "current_price":float(latest['Close']),
        "open":float(latest['Open']),
        "High":float(latest['High']),
        "Low":float(latest['Low']),
        "Volume":float(latest['Volume']),
    })

tools = [
    {
        "type":"function",
        "name":"get_korean_stock_price",
        "description":"한국 주식 종목코드를 입력받아 현재 가격을 조회합니다.",
        "parameters":{
            "type":"object",
            "properties":{
                "code":{
                    "type":"string",
                    "description":"6자리 한국 주식 종목 코드(예: 005930)"
                },               
            },
            "required":["code"],
            "additionalProperties":False
        },
        "strict":True
    }
]


def run_korean_stock_agent(user_input:str):

    response = client.responses.create(
        model=MODEL,
        input=user_input, 
        tools=tools  
    )

    function_calls = [item  for item in response.output if item.type == "function_call"]

    if not function_calls:
        return response.output_text


    tool_outputs = []
    for item in function_calls:
        # AI가 전달한 인자 가져오기
        args = json.loads(item.arguments) 
        # func 실행
        result = get_korean_stock_price(**args)

        tool_outputs.append(
            {
                "type":"function_call_output",
                "call_id":item.call_id,
                "output": result 
            }
        )


    final_response = client.responses.create(
        model=MODEL,
        previous_response_id=response.id,
        input=tool_outputs,  
        tools=tools  
    )

    return final_response.output_text    

In [20]:
print(run_korean_stock_agent("005930"))
print(run_korean_stock_agent("005930 주가 알려줘"))

삼성전자(005930) 현재가는 **212,750원**입니다.

시가 214,000원 · 고가 216,000원 · 저가 212,000원 · 거래량 4,040,543주
삼성전자(005930) 현재가는 **212,750원**입니다.  
시가 214,000원, 고가 216,000원, 저가 212,000원이며 거래량은 약 404만 주입니다.


In [21]:
# openai tools => web_search

response = client.responses.create(
    model=MODEL,
    input="오늘의 좋은 소식 알려줘", 
    tools=[{"type":"web_search"}]  
)

print(response.output_text)

오늘은 **2026년 11월 13일**이에요. 오늘 막 나온 소식만은 아니지만, 최근 며칠 사이 전해진 반가운 소식 세 가지를 골라봤어요.

- **사라질 뻔한 바닷새가 크게 늘었어요.** 칠레 코로스섬에서 페루다이빙슴새가 2013년 약 1만 2천 쌍에서 2024년 약 9만 쌍으로 증가했대요. 섬의 침입종 토끼를 없애고 서식지를 회복한 노력이 큰 몫을 했다고 합니다. ([islandconservation.org](https://www.islandconservation.org/press-release-new-study-reveals-300-fold-recovery-of-peruvian-diving-petrel-following-island-restoration-in-chile/?utm_source=openai))
- **나가족 조상 유해 39구가 공동체에 돌아가게 됐어요.** 옥스퍼드대학교는 11월 9일 나가족 대표들에게 유해의 관리권을 넘겼고, 고향 땅에 모실 계획이라고 밝혔어요. ([prm.ox.ac.uk](https://prm.ox.ac.uk/a-path-home?utm_source=openai))
- **여섯 살 아이와 염소들이 유기견을 도왔어요.** 반려견을 잃은 아이가 염소들과 모금에 나서 동물 보호소를 위해 1만 5,800달러를 모았다고 해요. ([goodnewsdose.com](https://goodnewsdose.com/boy-goats-raised-money-dog-shelters/?utm_source=openai))

작은 친절과 꾸준한 보살핌이 실제 변화를 만든다는 소식들이네요.


In [ ]:
response = client.responses.create(
    model=MODEL,
    input="오늘의 좋은 소식 알려줘", 
    tools=[{"type":"web_search", "filters":{
        "allowed_domains":[
            "www.naver.com",
        ],
        "blocked_domains":[
            "reddit.com",
            "wikipedia.com",
        ]
    }}]  
)

print(response.output_text)

In [22]:
# 이미지 웹 검색

response = client.responses.create(
    model=MODEL,
    reasoning={"effort":"low"},
    input="Search for recent images and supporting text sources about the Golden Gate Bridge at sunset.", 
    tools=[
        {
            "type":"web_search", 
            "search_content_types": ['image','text'],
            "image_settings":{
                "max_results":3,
                "caption":True
            }
        }
    ]  
)

print(response.output)

[ResponseFunctionWebSearch(id='ws_0c84f54cfb76ae85006af679ddb1f087d283ed3d1b51ebd901', action=ActionSearch(type='search', queries=['Golden Gate Bridge sunset photograph sunset source', 'Golden Gate Bridge sunset photos 2025'], query='Golden Gate Bridge sunset photograph sunset source', sources=None), status='completed', type='web_search_call'), ResponseOutputMessage(id='msg_0c84f54cfb76ae85006af679e4ffac87d294ceb6eba03ac0ec', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=465, start_index=346, title='File:Golden Gate at sunset - 2025.jpg - Wikimedia Commons', type='url_citation', url='https://commons.wikimedia.org/wiki/File%3AGolden_Gate_at_sunset_-_2025.jpg?utm_source=openai'), AnnotationURLCitation(end_index=808, start_index=686, title='Golden gate bridge at sunset through its tower. photo – Free Sunset Image on Unsplash', type='url_citation', url='https://unsplash.com/photos/golden-gate-bridge-at-sunset-through-its-tower-5bL0QZOkEHI?utm_source=openai'), Ann